# Modes of antiresonant fibers

```{index} antiresonant fiber
```
```{index} ARF; hollow-core fiber
```

*Antiresonant fibers* (ARF) are microstructured hollow-core fibers that confine light in an air core using a ring of thin glass capillary tubes. Light of free-space wavelength $\lambda$ impinging on a capillary wall of thickness $t$ and index $n$ is strongly transmitted (and thus lost from the core) at the *resonant* wavelengths

$$
\lambda_m = \frac{2t}{m}\sqrt{n^2-1}, \qquad m = 1, 2, \ldots,
$$

and strongly reflected in the *antiresonant* windows between them — the ARROW (antiresonant reflecting optical waveguide) mechanism. Since the core is air, such fibers offer low nonlinearity and high damage thresholds; their price is that all core modes are *leaky*: as in the [step-index](./1_3_stepindex_leaky.ipynb) and [Bragg](./2_2_bragg.ipynb) notebooks, propagation constants $\beta$ are complex, and the confinement loss (CL) in dB/m is computable from $\operatorname{Im}\beta$.

The `ARF` class provides the six-capillary geometry of [[Poletti (2014)](#references)] by default, and the eight-capillary geometry of [[Kolyadin et al (2013)](#references)] by `name='kolyadin'`.

In [ ]:
import ngsolve as ng
import numpy as np
from ngsolve.webgui import Draw
from fibermode import ARF

## Constructing `ARF` objects

The constructor builds the geometry and mesh. By default the capillary tubes are *embedded* into the glass cladding sheath (pass `freecapil=True` for the unphysical free-standing tubes geometry). Updatable lengths (core radius, capillary radii and thickness, cladding and PML thicknesses) can be overridden by keyword arguments in micrometers.

In [ ]:
a = ARF()   # 6-tube Poletti geometry, embedded capillaries
print(a)

In [ ]:
Draw(a.mesh);

The operating wavelength is an attribute (1800 nm by default for this geometry); resetting it updates the nondimensional material coefficients automatically.

In [ ]:
a.wavelength

The constructor of ARF takes many more arguments, including a generic `**kwargs`, so your best bet to find out how to tweak the geometry defaults is by following the constructor's docstring comments, which you can view using `ARF.__init__?`.

```{index} ARF; keyword arguments 
```

## Computing leaky modes

As in the Bragg fiber notebook, we use the FEAST eigensolver together with a *frequency-dependent* PML, which poses the leaky mode problem as a polynomial eigenproblem in the nondimensional resonance value $Z$ (not $Z^2$). The contour below encloses the $Z$-value of the LP01-like fundamental core mode of this geometry.

```{index} leakymode; ARF
```

In [ ]:
p = 2       # finite element degree
Z, y, yl, beta, P, extras = a.leakymode(
    p,
    ctr=2.24,       # contour center in the Z-plane
    rad=0.02,       # contour radius
    alpha=5,        # PML strength
    nspan=4,
    npts=4,
    seed=1,
    niterations=50,
    nrestarts=0,
)

The physical propagation constants and confinement loss follow from the computed $Z$: 

In [ ]:
print('beta      =', beta)
print('CL [dB/m] =', 20 * beta.imag / np.log(10))

The fundamental mode intensity is concentrated in the hollow core, shielded by the capillary ring:

In [ ]:
Draw(ng.Norm(y.gridfun())**2, a.mesh);

## Higher-order core modes

For this geometry at 1800 nm, the following contour centers (radius 0.02) capture the labeled mode groups:

| Mode  | $Z$-center |
|-------|-----------|
| LP01  | 2.24      |
| LP11  | 3.57      |
| LP21  | 4.75      |
| LP02  | 5.09      |

For example, the LP11-like mode group:

In [ ]:
Z11, y11, yl11, beta11, P11, _ = a.leakymode(
    p, ctr=3.57, rad=0.02, alpha=5, nspan=4, npts=4,
    seed=1, niterations=50, nrestarts=0)
print('CL [dB/m] =', 20 * beta11.imag / np.log(10))

In [ ]:
Draw(ng.Norm(y11.gridfun())**2, a.mesh);

Note how much lossier the higher-order modes are than the fundamental: this differential loss is exactly what makes ARFs behave as effectively single-mode fibers over long lengths.

## Cross-checking with other formulations

The same modes can be computed by the standard frequency-independent PML, which yields a *linear* eigenproblem in $Z^2$ (so contour parameters must be specified in the $Z^2$-plane — a factor worth double-checking whenever results look off):

In [ ]:
zsqr, Y2, Yl2, beta2, P2 = a.leakymode_auto(
    p, centerZ2=2.24**2, radiusZ2=0.1, alpha=5,
    npts=4, nspan=4, seed=1, niterations=50, nrestarts=0)
print('beta (auto PML) =', beta2)

Agreement of `beta` between the two formulations (to a tolerance consistent with the discretization) is a useful sanity check, since the two solvers use different PMLs and different eigensolver linearizations.

## The eight-capillary fiber of Kolyadin et al

The `name='kolyadin'` geometry has eight, larger, capillaries. For finite element degrees $p \geq 3$, refine the mesh at least once, and use the mode table from `demos/arf/arf_kolyadin_mode.py` (LP01 at $Z \approx 2.29$):

In [ ]:
b = ARF(name='kolyadin', 
        capillary_maxhs=0.2, 
        freecapil=False)
b.refine()
Draw(b.mesh);

In [ ]:
Zk, yk, ylk, betak, Pk, _ = b.leakymode(
    3, ctr=2.29, rad=0.02, alpha=5, nspan=4, npts=4,
    seed=1, niterations=50, nrestarts=0)
k = 2 * np.pi / b.wavelength
print('effective index =', betak / k)

In [ ]:
Draw(ng.Norm(yk.gridfun())**2, b.mesh);

<a id='references'></a>
## References

- F. Poletti. *Nested antiresonant nodeless hollow core fiber.* Optics Express, 22:23807-23828, 2014.
- A. D. Kolyadin, A. F. Kosolapov, A. D. Pryamikov, A. S. Biriukov, V. G. Plotnichenko, and E. M. Dianov. *Light transmission in negative curvature hollow core fiber in extremely high material loss region.* Optics Express, 21:9514-9519, 2013.
- W. Belardi and J. C. Knight. *Hollow antiresonant fibers with reduced attenuation.* Optics Letters, 39:1853-1856, 2014.